**Introduction**
//Évaluation de l'Extracteur de Caractéristiques (Backbone)//

Ce notebook est dédié à l'analyse et à la validation du Backbone, composant essentiel de notre architecture d'estimation d'âge étudiée dans ce projet. L'objectif principal est de vérifier la pertinence des descripteurs visuels extraits par ce modèle avant son utilisation comme socle.


**Objectif du Notebook** 

L'enjeu de cette phase est de valider la qualité du modèle. Un bon extracteur de caractéristiques doit être capable de :

- Généraliser : Produire des descripteurs cohérents sur des visages issus de conditions variées (comme pour le dataset APPA-REAL que nous utiliserons pour l'estimation d'âge).

- Discriminer : Séparer efficacement les attributs faciaux dans l'espace latent.

Cette validation est une étape préliminaire indispensable : la performance de toute tête de prédiction ajoutée par la suite dépendra directement de la richesse sémantique de ce Backbone.

In [ ]:
# ============================================================
# Cellule 1 : Setup et repo
# ============================================================
import os, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

repo_url = "https://github.com/Lduvignacq/Deep-learning-project.git"
repo_dir = "Deep-learning-project"

if not os.path.exists(repo_dir):
    print(f"Clonage du repo : {repo_url}")
    !git clone {repo_url}
else:
    print(f"Le dossier {repo_dir} existe déjà.")

os.chdir(repo_dir)
print(f"Répertoire de travail : {os.getcwd()}")


In [ ]:
# ============================================================
# Cellule 2 : Dépendances
# ============================================================
!pip install pretrainedmodels seaborn -q


In [ ]:
# ============================================================
# Cellule 3 : Chargement du backbone se_resnext50 (ImageNet)
# ============================================================
import torch.nn as nn
import sys
sys.path.insert(0, os.getcwd())
from model import get_model

backbone = get_model(model_name="se_resnext50_32x4d", num_classes=101, pretrained="imagenet")
backbone.last_linear = nn.Identity()  # on coupe la tête de classification ImageNet
backbone = backbone.to(device)
backbone.eval()

# Backbone complètement gelé — on ne touche pas aux poids
for param in backbone.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in backbone.parameters())
print(f"✓ Backbone chargé ({total_params:,} paramètres, tous gelés)")


In [ ]:
# ============================================================
# Cellule 4 : Le backbone reconnaît-il les visages ? (sans entraînement)
# ============================================================
!pip install scikit-learn -q

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.datasets import fetch_lfw_people
from PIL import Image

# --- Téléchargement LFW via sklearn ---
print("⬇️  Téléchargement LFW (visages) via sklearn...")
lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True)
lfw_images = lfw.images
print(f"  LFW : {len(lfw_images)} images")

print("⬇️  Téléchargement CIFAR-10 (non-visages)...")
cifar_split_a = datasets.CIFAR10(root="./data", download=True, train=True)
cifar_split_b = datasets.CIFAR10(root="./data", download=True, train=False)
print(f"  CIFAR-10 : {len(cifar_split_a) + len(cifar_split_b)} images au total")

# --- Transform ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# --- Datasets ---
class LFWDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images    = images
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = (self.images[idx] * 255).astype(np.uint8)
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, 1

class CIFARDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset   = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        if self.transform:
            img = self.transform(img)
        return img, 0

lfw_ds     = LFWDataset(lfw_images, transform=transform)
cifar_full = ConcatDataset([
    CIFARDataset(cifar_split_a, transform=transform),
    CIFARDataset(cifar_split_b, transform=transform),
])

n = min(len(lfw_ds), len(cifar_full))
print(f"\n✓ {n} visages + {n} non-visages → {n*2} images")

lfw_ds_sub,   _ = random_split(lfw_ds,     [n, len(lfw_ds)    - n])
cifar_ds_sub, _ = random_split(cifar_full, [n, len(cifar_full) - n])

full   = ConcatDataset([lfw_ds_sub, cifar_ds_sub])
loader = DataLoader(full, batch_size=128, shuffle=False, num_workers=2)

# --- Extraction des features ---
print("\n⚙️  Extraction des features backbone...")
backbone.eval()
all_feats, all_labels, all_imgs_raw = [], [], []

# Transform sans normalisation pour l'affichage visuel
transform_display = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

with torch.no_grad():
    for i, (imgs, labels) in enumerate(loader):
        imgs  = imgs.to(device)
        feats = backbone(imgs).cpu().numpy()
        all_feats.append(feats)
        all_labels.append(labels.numpy())
        if i % 20 == 0:
            print(f"  batch {i}/{len(loader)}...")

all_feats  = np.concatenate(all_feats,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
all_feats  = normalize(all_feats, norm="l2")
print(f"  Features extraites : {all_feats.shape}")

# --- K-means ---
print("\n⚙️  K-means (k=2)...")
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(all_feats)

acc_direct = np.mean(cluster_labels == all_labels)
acc_flip   = np.mean((1 - cluster_labels) == all_labels)
if acc_flip > acc_direct:
    cluster_labels = 1 - cluster_labels

# --- Matrice de confusion ---
TP = int(((cluster_labels == 1) & (all_labels == 1)).sum())
TN = int(((cluster_labels == 0) & (all_labels == 0)).sum())
FP = int(((cluster_labels == 1) & (all_labels == 0)).sum())
FN = int(((cluster_labels == 0) & (all_labels == 1)).sum())

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

conf_matrix = np.array([[TP, FN], [FP, TN]])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    conf_matrix,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Prédit : Visage", "Prédit : Non-visage"],
    yticklabels=["Réel : Visage",   "Réel : Non-visage"],
    ax=ax
)
ax.set_title(
    f"Matrice de confusion — Backbone ImageNet brut (k-means)\n"
    f"N={n*2} | Accuracy : {accuracy:.1%} | Précision : {precision:.1%} | Rappel : {recall:.1%}",
    fontsize=11
)
plt.tight_layout()
plt.show()

print(f"\n  TP : {TP} | TN : {TN} | FP : {FP} | FN : {FN}")
print(f"  Accuracy : {accuracy:.1%} | Précision : {precision:.1%} | Rappel : {recall:.1%}")
print(f"  (baseline aléatoire : 50%)")

if accuracy > 0.85:
    print("\n✅ Le backbone sépare naturellement les visages — features très discriminantes.")
elif accuracy > 0.70:
    print("\n✓  Séparation partielle — le backbone voit une différence mais elle n'est pas nette.")
else:
    print("\n⚠️  Pas de séparation claire (~50% = aléatoire).")


In [ ]:
# ── Visualisation 4x4 améliorée : Matrice de Confusion ──────────────────

N_SHOW = 4

# Indices pour chaque case de la matrice de confusion
idx_TP = np.where((cluster_labels == 1) & (all_labels == 1))[0]  # vrais positifs
idx_TN = np.where((cluster_labels == 0) & (all_labels == 0))[0]  # vrais négatifs
idx_FP = np.where((cluster_labels == 1) & (all_labels == 0))[0]  # faux positifs
idx_FN = np.where((cluster_labels == 0) & (all_labels == 1))[0]  # faux négatifs

# Petit helper pour reconstruire une image affichable à partir du dataset "full"
# (qui contient des images normalisées ImageNet)
def get_image_from_full(dataset, idx):
    img, _ = dataset[idx]           # img : tensor (C, H, W) normalisé
    if isinstance(img, torch.Tensor):
        img = img.clone()

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

        img = img * std + mean      # dé-normalisation
        img = torch.clamp(img, 0, 1)
        img_np = img.permute(1, 2, 0).numpy()
    else:
        # au cas où le dataset renverrait un numpy/PIL (normalement non ici)
        img_np = np.array(img)

    return img_np

# Définition des étiquettes détaillées pour chaque ligne
cases = [
    (idx_TP, "VRAIS POSITIFS (TP)\n\nRéalité : Visage (LFW)\nPrédit : Visage", "green"),
    (idx_TN, "VRAIS NÉGATIFS (TN)\n\nRéalité : Objet (CIFAR)\nPrédit : Objet", "green"),
    (idx_FP, "FAUX POSITIFS (FP)\n\nRéalité : Objet (CIFAR)\nPrédit : Visage", "red"),
    (idx_FN, "FAUX NÉGATIFS (FN)\n\nRéalité : Visage (LFW)\nPrédit : Objet", "red"),
]

fig, axes = plt.subplots(4, N_SHOW, figsize=(N_SHOW * 2.2, 4 * 3))

for row, (indices, title, color) in enumerate(cases):
    # Label textuel à gauche de la première image de chaque ligne
    axes[row, 0].set_ylabel(
        title,
        fontsize=9,
        fontweight="bold",
        rotation=0,
        ha="right",
        va="center",
        color=color,
    )

    if len(indices) == 0:
        # Si aucun exemple pour cette case, on remplit la ligne avec un message
        for col in range(N_SHOW):
            ax = axes[row, col]
            ax.text(0.5, 0.5, "Aucun exemple trouvé", ha="center", va="center")
            ax.axis("off")
        continue

    # Sélection aléatoire des indices à afficher
    sample = np.random.choice(indices, size=min(N_SHOW, len(indices)), replace=False)

    for col in range(N_SHOW):
        ax = axes[row, col]
        if col < len(sample):
            img_np = get_image_from_full(full, sample[col])
            ax.imshow(img_np)
            # Bordure colorée pour rappeler la case (TP/TN/FP/FN)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
                spine.set_visible(True)
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
            ax.axis("off")

        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle(
    "Analyse qualitative de l'extracteur de caractéristiques\n"
    "(Matrice de confusion : Visages vs Objets)",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)

# On ajuste les marges pour laisser de la place au texte à gauche
plt.subplots_adjust(left=0.30, wspace=0.05, hspace=0.25)
plt.show()